# Прототип: обнаружение и локализация опухолей мозга на МРТ (демо)
**Запуск:** «Среда выполнения» → «Выполнить все». Видеокарта не обязательна (работает и на CPU, ~2–3 минуты на установку).
В конце появится ссылка `https://….gradio.live`: откройте её, загрузите срез МРТ (JPG/PNG) или выберите пример.

Модели уже обучены и лежат в репозитории (`weights/`); полный код обучения и проверок — `Almazov_MRI_v3.ipynb`.
Исследовательский прототип, **не медицинское изделие**. Решение принимает врач.

In [ ]:
# === 1. Установка и загрузка обученных моделей (из этого репозитория) ===
!pip -q install timm segmentation-models-pytorch gradio
import os, subprocess, hashlib, json
REPO = 'https://github.com/blond7I/almazov-mri-tumor'
if not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, 'repo'], check=True)
os.makedirs('weights', exist_ok=True)
if not os.path.exists('weights/seg.pt'):   # сегментация лежит в репозитории частями по 20 МБ
    with open('weights/seg.pt', 'wb') as out:
        for i in range(5): out.write(open(f'repo/weights/seg_v3_best.pt.part{i}', 'rb').read())
h = hashlib.sha256(open('weights/seg.pt', 'rb').read()).hexdigest()
assert h.startswith('ac9d92e985ae'), 'Файл сегментации собрался неверно'
# классификатор: финальная модель (выбрана по валидации) описана в weights/cls_models.json; если файла нет — основная EfficientNet-B0
MAN = 'repo/weights/cls_models.json'
if os.path.exists(MAN):
    CLS = json.load(open(MAN))
    for m in CLS['models']:
        dst = f"weights/cls_{m['tag']}.pt"
        if not os.path.exists(dst):
            with open(dst, 'wb') as out:
                for prt in m['parts']: out.write(open('repo/weights/' + prt, 'rb').read())
        assert hashlib.sha256(open(dst, 'rb').read()).hexdigest() == m['sha256'], 'Файл модели собрался неверно: ' + dst
        m['file'] = dst
else:
    if not os.path.exists('weights/cls.pt'):
        open('weights/cls.pt', 'wb').write(open('repo/weights/cls_v3_best.pt', 'rb').read())
    CLS = dict(selected='base', screen_thr=0.5, models=[dict(tag='base', arch='efficientnet_b0', size=256, file='weights/cls.pt')])
print('Классификатор:', CLS['selected'], '->', [m['arch'] for m in CLS['models']])
print('Модели готовы:', {f: round(os.path.getsize('weights/' + f) / 1e6, 1) for f in os.listdir('weights')}, 'МБ')

In [ ]:
# === 2. Модели и логика анализа ===
import glob, contextlib, numpy as np, cv2, torch, timm, segmentation_models_pytorch as smp, gradio as gr
from PIL import Image
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
amp = (lambda: torch.autocast('cuda')) if dev == 'cuda' else contextlib.nullcontext
CLASSES = ['Глиома', 'Менингиома', 'Опухоль гипофиза', 'Опухоли нет']
MEAN, STD = 0.449 * 255, 0.226 * 255
THR = float(CLS.get('screen_thr', 0.5))   # порог «есть опухоль» (режим скрининга), выбран по валидации

models = []
for m in CLS['models']:
    assert m.get('size', 256) == 256, 'демо рассчитано на вход 256x256'
    net = timm.create_model(m['arch'], pretrained=False, num_classes=4)
    net.load_state_dict(torch.load(m['file'], map_location='cpu')); models.append(net.to(dev).eval())
seg = smp.Unet('resnet34', encoder_weights=None, in_channels=1, classes=1)
seg.load_state_dict(torch.load('weights/seg.pt', map_location='cpu')); seg.to(dev).eval()

def prep(g):
    # та же предобработка, что при обучении: серый -> квадрат -> 256x256 -> контраст по перцентилям
    h, w = g.shape; s = max(h, w)
    if h != w:
        c = np.zeros((s, s), g.dtype); c[(s-h)//2:(s-h)//2+h, (s-w)//2:(s-w)//2+w] = g; g = c
    g = cv2.resize(g, (256, 256), interpolation=cv2.INTER_AREA if s > 256 else cv2.INTER_LINEAR).astype(np.float32)
    lo, hi = np.percentile(g, 0.5), np.percentile(g, 99.5)
    return (np.clip((g - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

@torch.no_grad()
def predict(x256):
    x = torch.tensor((x256.astype(np.float32) - MEAN) / STD)[None, None].to(dev)
    with amp():
        x3 = x.repeat(1, 3, 1, 1)   # среднее по моделям ансамбля и по отражению (как при оценке)
        p = torch.stack([(net(x3).float().softmax(1) + net(torch.flip(x3, (3,))).float().softmax(1)) / 2 for net in models]).mean(0)[0].cpu().numpy()
        pm = ((torch.sigmoid(seg(x).float()) + torch.flip(torch.sigmoid(seg(torch.flip(x, (3,))).float()), (3,))) / 2)[0, 0].cpu().numpy()
    return p, pm

def place(m):
    ys, xs = np.nonzero(m); cy, cx = ys.mean() / 256, xs.mean() / 256
    v = 'верхней' if cy < .4 else ('нижней' if cy > .6 else 'средней')
    h = 'левой' if cx < .4 else ('правой' if cx > .6 else 'центральной')
    return f'в {v} {h} части изображения'

def core(img):
    # img: RGB/серый numpy -> вероятности, оценка «есть опухоль», маска, картинка с контуром, пояснение
    x256 = prep(np.array(Image.fromarray(img).convert('L')))
    p, pm = predict(x256); p_t = 1 - p[3]
    m = (pm > .5).astype(np.uint8)
    k, lab, st, _ = cv2.connectedComponentsWithStats(m)
    for c in range(1, k):
        if st[c, cv2.CC_STAT_AREA] < 30: m[lab == c] = 0
    rgb = cv2.cvtColor(x256, cv2.COLOR_GRAY2RGB)
    lines = [f'Оценка модели «есть опухоль»: {p_t:.0%} (порог {THR}).']
    if m.sum():
        ov = rgb.copy(); ov[m > 0] = (255, 0, 0); rgb = cv2.addWeighted(rgb, .65, ov, .35, 0)
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(rgb, cs, -1, (255, 255, 0), 1)
    if p_t >= THR:
        top = int(np.argmax(p[:3])); lines.append(f'Наиболее вероятный тип: {CLASSES[top]} ({p[top]:.0%}).')
        if m.sum():
            lines.append(f'Выделена область: {100*m.mean():.1f}% площади среза, {place(m)}. Подозрение подтверждено найденной областью.')
        else:
            lines.append('Область не выделена: подозрение НЕ подтверждено сегментацией — нужен внимательный просмотр врачом.')
    else:
        lines.append('Признаков опухоли не найдено.' if not m.sum() else 'Классификатор опухоли не видит, но сегментация выделила область — проверьте отмеченный участок.')
    if p.max() < .6 or .3 < p_t < .7: lines.append('ВНИМАНИЕ: низкая уверенность модели — приоритетный просмотр врачом.')
    lines.append('Исследовательский прототип, не медицинское изделие. Решение принимает врач.')
    return p, p_t, m, rgb, lines

def analyze(img):
    if img is None: return None, None, 'Загрузите снимок МРТ (JPG/PNG).'
    p, p_t, m, rgb, lines = core(img)
    return {CLASSES[i]: float(p[i]) for i in range(4)}, rgb, '\n'.join(lines)

examples = sorted(glob.glob('repo/examples/*.jpg'))[:8]
gr.Interface(analyze, gr.Image(type='numpy', label='Срез МРТ (JPG/PNG)'),
             [gr.Label(num_top_classes=4, label='Оценки модели'), gr.Image(label='Результат (красным — найденная область)'),
              gr.Textbox(label='Пояснение для врача', lines=6)],
             examples=examples or None,
             title='МРТ головного мозга: обнаружение и локализация опухоли',
             description='Загрузите срез МРТ. Система оценит наличие и тип опухоли и выделит её область. Решение принимает врач.'
             ).launch(share=True, debug=False)

## Проверка на своих снимках (пакетно)
Положите снимки в папку (можно архивом `.zip` — загрузите его через значок папки слева), укажите путь в `FOLDER` и запустите ячейку.
Если снимки разложены по папкам классов (`glioma/`, `meningioma/`, `pituitary/`, `notumor/` или `Глиома/`, `Менингиома/`, `Гипофиз/` или `Опухоль гипофиза/`, `Нет опухоли/`) — ячейка сама посчитает точность.
Результат: таблица `batch_results.csv` и картинки с контуром в `batch_overlays/`. По умолчанию проверяются примеры из репозитория.

In [ ]:
# === 3. Пакетная проверка папки со снимками ===
import zipfile, shutil, pandas as pd
FOLDER = 'repo/examples'          # <- путь к своей папке или .zip
def unzip(path, dst):
    # архивы из Windows хранят русские имена не в UTF-8 — чиним кодировку
    shutil.rmtree(dst, ignore_errors=True)
    with zipfile.ZipFile(path) as z:
        for info in z.infolist():
            name = info.filename
            if not info.flag_bits & 0x800:
                raw = name.encode('cp437')
                try: name = raw.decode('utf-8')
                except UnicodeDecodeError: name = raw.decode('cp866')
            target = os.path.join(dst, name)
            if info.is_dir(): os.makedirs(target, exist_ok=True); continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as out: out.write(src.read())
if FOLDER.endswith('.zip'):
    unzip(FOLDER, 'batch_input'); FOLDER = 'batch_input'
KEYS = {0: ['glioma', 'глиома'], 1: ['meningioma', 'менингиома'], 2: ['pituitary', 'гипофиз', 'опухоль гипофиза'],
        3: ['notumor', 'no_tumor', 'no tumor', 'normal', 'нет опухоли', 'норма']}
def true_label(path):
    # класс берём из имени папки, иначе из начала имени файла; не нашли -> None
    for part in [os.path.basename(os.path.dirname(path)), os.path.basename(path)]:
        t = part.strip().lower()
        for c, ks in KEYS.items():
            if any(t.startswith(k) for k in ks): return c
    return None
files = sorted(f for f in glob.glob(os.path.join(FOLDER, '**', '*'), recursive=True)
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')))
shutil.rmtree('batch_overlays', ignore_errors=True); os.makedirs('batch_overlays'); rows = []
for i, f in enumerate(files):
    p, p_t, m, rgb, _ = core(np.array(Image.open(f).convert('RGB')))
    pred = int(np.argmax(p[:3])) if p_t >= THR else 3
    y = true_label(f)
    out = f'batch_overlays/{i:04d}_{os.path.basename(f)}'.rsplit('.', 1)[0] + '.png'
    cv2.imwrite(out, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    rows.append({'file': os.path.relpath(f, FOLDER), 'prediction': CLASSES[pred], 'p_tumor': round(float(p_t), 3),
                 **{f'p_{CLASSES[c]}': round(float(p[c]), 3) for c in range(4)},
                 'contour_found': bool(m.sum()), 'area_%': round(100 * float(m.mean()), 2),
                 'true': CLASSES[y] if y is not None else '', 'correct': (pred == y) if y is not None else ''})
df = pd.DataFrame(rows); df.to_csv('batch_results.csv', index=False, encoding='utf-8-sig')
print(f'Обработано снимков: {len(df)}. Таблица: batch_results.csv, картинки: batch_overlays/')
lab = df[df.true != '']
if len(lab):
    yt, yp = (lab.true != 'Опухоли нет'), (lab.prediction != 'Опухоли нет')
    print(f'Размечено: {len(lab)}. Точность (4 класса): {(lab.prediction == lab.true).mean():.1%}')
    if yt.sum(): print(f'Чувствительность «опухоль есть»: {(yt & yp).sum() / yt.sum():.1%}')
    if (~yt).sum(): print(f'Специфичность: {(~yt & ~yp).sum() / (~yt).sum():.1%}')
    print(pd.crosstab(lab.true, lab.prediction, rownames=['истина'], colnames=['модель']))
df.head(20)